In [ ]:
from collections import defaultdict
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import numpy as np
import xarray as xr
from context_flux_no.metrics import relative_L2_error, relative_L_infty_error
from context_flux_no.training.io import load_model
from einops import rearrange
from more_itertools import one
from tqdm import tqdm


datadir = Path("../../data")
checkpoint_dir = Path("../../checkpoints")

jax.config.update("jax_default_device", jax.devices("gpu")[0])

In [ ]:
@eqx.filter_jit
def compute_metrics(model, u, args, context_length=20):
    context, u_data = u[:context_length], u[context_length:]

    u_pred = model.rollout(context, args, num_steps=len(u_data))[0]

    return {
        "l2_onestep": relative_L2_error(u_pred[0], u_data[0]),
        "l_inf_onestep": relative_L_infty_error(u_pred[0], u_data[0]),
        "l2_rollout": relative_L2_error(u_pred, u_data),
        "l_inf_rollout": relative_L_infty_error(u_pred, u_data),
    }

## In-distribution results

In [ ]:
dataset_test = xr.open_dataset(
    datadir / "datasets/burgers_1d/data/test/burgers_1d_test_grf_seed=10.hdf5",
    engine="h5netcdf",
    chunks={},
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = dataset_test["values"].values
segments = np.lib.stride_tricks.sliding_window_view(values, 40, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

In [ ]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])
results_dict = defaultdict(list)
for model_type in ("NaiveFluxNO",):  # ("DPOT", "HyperFluxFNOLocal", "DISCO"):
    for seed in tqdm((0, 10, 20)):
        model_dir = (
            checkpoint_dir / "burgers_1d" / model_type / "OneStepLoss" / f"seed={seed}"
        )
        model = load_model(one(model_dir.iterdir()))
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(lambda u: compute_metrics(model, u, (dt, dx))),
            segments,
            batch_size=1500,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[model_type].append(jax.tree.map(jnp.mean, results))

In [ ]:
import json


with open("../../data/analysis/burgers_indistribution.json", "w") as f:
    json.dump(jax.tree.map(lambda x: float(x), results_dict), f)

In [ ]:
for model_name, res in results_dict.items():
    res_ = jax.tree.transpose(jax.tree.structure(["*"] * 3), None, res)
    stats = jax.tree.map(
        lambda list_: {
            "mean": jnp.mean(jnp.asarray(list_)),
            "std": jnp.std(jnp.asarray(list_)),
        },
        res_,
        is_leaf=lambda x: isinstance(x, list),
    )
    for k, v in stats.items():
        print(model_name, f"{k}: {v['mean']:.2e}" + "±" + f"{v['std']:.2e}")

## OOD results

In [ ]:
dataset_test = xr.open_dataset(
    datadir / "datasets/burgers_1d/data/test/burgers_1d_test_step_seed=10.hdf5",
    engine="h5netcdf",
    chunks={},
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = dataset_test["values"].values
segments = np.lib.stride_tricks.sliding_window_view(values, 40, axis=1)
segments = rearrange(segments, "batch t0 c x t -> (batch t0) t c x")
segments.shape

In [ ]:
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])
results_dict = defaultdict(list)
for model_type in ("DPOT", "HyperFluxFNOLocal", "DISCO"):
    for seed in tqdm((0, 10, 20)):
        model_dir = (
            checkpoint_dir / "burgers_1d" / model_type / "OneStepLoss" / f"seed={seed}"
        )
        model = load_model(one(model_dir.iterdir()))
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(lambda u: compute_metrics(model, u, (dt, dx))),
            segments,
            batch_size=4000,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[model_type].append(jax.tree.map(jnp.mean, results))

import json


with open("../../data/analysis/burgers_ood_step.json", "w") as f:
    json.dump(jax.tree.map(lambda x: float(x), results_dict), f)

In [ ]:
import json


with open("../../data/analysis/burgers_ood_step.json", "r") as f:
    results_dict = json.load(f)

In [ ]:
for model_name, res in results_dict.items():
    res_ = jax.tree.transpose(jax.tree.structure(["*"] * 3), None, res)
    print(
        model_name,
        jax.tree.map(
            lambda list_: {
                "mean": jnp.mean(jnp.asarray(list_)),
                "std": jnp.std(jnp.asarray(list_)),
            },
            res_,
            is_leaf=lambda x: isinstance(x, list),
        ),
    )

## Long time

In [ ]:
@eqx.filter_jit
def metrics_long_rollout(model, u, args, context_length=20):
    context, u_data = u[:context_length], u[context_length:]
    u_pred = model.rollout(context, args, num_steps=len(u_data))[0]
    l2 = jax.vmap(relative_L2_error)(u_pred, u_data)
    l_inf = jax.vmap(relative_L_infty_error)(u_pred, u_data)
    return {"l2": l2, "l_inf": l_inf}

In [ ]:
dataset_test = xr.open_dataset(
    datadir / "datasets/burgers_1d/data/test/burgers_1d_test_grf_seed=10.hdf5",
    engine="h5netcdf",
    chunks={},
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = dataset_test["values"].values

In [ ]:
results_dict = defaultdict(list)
for model_type in ("DPOT", "HyperFluxFNOLocal", "DISCO"):
    for seed in tqdm((0, 10, 20)):
        model_dir = (
            checkpoint_dir / "burgers_1d" / model_type / "OneStepLoss" / f"seed={seed}"
        )
        model = load_model(one(model_dir.iterdir()))
        model = eqx.nn.inference_mode(model, True)
        results = jax.lax.map(
            eqx.filter_jit(lambda u: metrics_long_rollout(model, u, (dt, dx))),
            values,
            batch_size=1000,
        )
        print(jax.tree.map(lambda x: x.shape, results))
        results_dict[model_type].append(results)

In [ ]:
results_stacked = {
    k: jax.tree.transpose(
        jax.tree.structure(["*"] * 3),
        None,
        v,
    )
    for k, v in results_dict.items()
}
results_stacked = jax.tree.map(
    np.stack, results_stacked, is_leaf=lambda x: isinstance(x, list)
)

with h5py.File("../../data/analysis/burgers_1d_long_rollout.hdf5", "w") as f:
    for model_name, res_dict in results_stacked.items():
        g = f.create_group(model_name)
        for metric_name, val in res_dict.items():
            g.create_dataset(metric_name, data=val)

In [ ]:
dataset_test = xr.open_dataset(
    datadir / "datasets/burgers_1d/data/test/burgers_1d_test_step_seed=10.hdf5",
    engine="h5netcdf",
    chunks={},
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = dataset_test["values"].values


u_pred_dict = {}
seed = 20
traj_idx = 5
for model_type in ("DPOT", "HyperFluxFNOLocal", "DISCO"):
    model_dir = (
        checkpoint_dir / "burgers_1d" / model_type / "OneStepLoss" / f"seed={seed}"
    )
    model = load_model(one(model_dir.iterdir()))
    model = eqx.nn.inference_mode(model, True)
    context, u_data = values[traj_idx, :20], values[traj_idx, 20:]
    u_pred = model.rollout(context, (dt, dx), num_steps=len(u_data))[0]

    u_pred_dict[model_type] = u_pred

In [ ]:
import colorcet as cc
import matplotlib.pyplot as plt


plt.style.use("context_flux_no.plotting.styles.dash_gridded")

COLOR_DICT = {
    "DPOT": cc.cm.glasbey_bw(85),
    "ICON": cc.cm.glasbey_bw(1),
    "DISCO": cc.cm.glasbey_bw(68),
    "HyperFluxFNOLocal": cc.cm.glasbey_bw(41),
}

LABEL_DICT = {
    "DPOT": "DPOT",
    "ICON": "ICON",
    "DISCO": "DISCO",
    "HyperFluxFNOLocal": "HFluxNO (Ours)",
}

fig, axes = plt.subplots(
    2,
    5,
    figsize=(9, 2.5),
    constrained_layout=True,
    sharex=True,
    sharey="row",
    height_ratios=(2, 1),
)
for i in range(axes.shape[1]):
    t_idx = 4 * 2**i - 1
    axes[0, i].plot(
        dataset_test.x,
        u_data[t_idx, 0],
        label="Data",
        color="black",
        linewidth=2.5,
        ls="--",
        alpha=0.6,
    )
    for model_type, u_pred in u_pred_dict.items():
        axes[0, i].plot(
            dataset_test.x,
            u_pred[t_idx, 0],
            label=LABEL_DICT[model_type],
            color=COLOR_DICT[model_type],
            linewidth=1.5,
            alpha=0.8,
        )
        axes[1, i].plot(
            dataset_test.x,
            u_pred[t_idx, 0] - u_data[t_idx, 0],
            label=LABEL_DICT[model_type],
            color=COLOR_DICT[model_type],
            linewidth=1.5,
            alpha=0.8,
        )
        axes[0, i].set_title("$t_{rollout}=$" + f"${t_idx * dt}$", fontsize="medium")
        # axes[0, i].text(0.4, -1.2, )
fig.supxlabel("$x$", fontsize="medium")
axes[0, 0].set_ylabel("$u_t(x)$")
axes[1, 0].set_ylabel("Error")
fig.align_ylabels()
fig.savefig("../../figures/fig2/rollout_trajectory_burgers_step.pdf", dpi=500)

## High resolution

In [ ]:
dataset_test = xr.open_dataset(
    datadir / "datasets/burgers_1d/data/test/burgers_1d_test_grf_hires_seed=10.hdf5",
    engine="h5netcdf",
    chunks={},
)
dt = float(dataset_test["t"][1] - dataset_test["t"][0])
dx = float(dataset_test["x"][1] - dataset_test["x"][0])

values = dataset_test["values"].values
values.shape

In [ ]:
u_pred_dict = {}
downsample = 2
seed = 0
traj_idx = 100
for model_type in ("HyperFluxFNOLocal", "DISCO"):
    model_dir = (
        checkpoint_dir / "burgers_1d" / model_type / "OneStepLoss" / f"seed={seed}"
    )
    model = load_model(one(model_dir.iterdir()))
    model = eqx.nn.inference_mode(model, True)
    context, u_data = (
        values[traj_idx, :20, :, ::downsample],
        values[traj_idx, 20:, :, ::downsample],
    )
    u_pred = model.rollout(context, (dt, dx * downsample), num_steps=len(u_data))[0]

    u_pred_dict[model_type] = u_pred

In [ ]:
import colorcet as cc
import matplotlib.pyplot as plt


plt.style.use("context_flux_no.plotting.styles.dash_gridded")

COLOR_DICT = {
    "DPOT": cc.cm.glasbey_bw(85),
    "ICON": cc.cm.glasbey_bw(1),
    "DISCO": cc.cm.glasbey_bw(68),
    "HyperFluxFNOLocal": cc.cm.glasbey_bw(41),
}

LABEL_DICT = {
    "DPOT": "DPOT",
    "ICON": "ICON",
    "DISCO": "DISCO",
    "HyperFluxFNOLocal": "HFluxNO (Ours)",
}

fig, axes = plt.subplots(
    2,
    5,
    figsize=(9, 2.5),
    constrained_layout=True,
    sharex=True,
    sharey="row",
    height_ratios=(2, 1),
)
for i in range(axes.shape[1]):
    t_idx = 4 * 2**i - 1
    axes[0, i].plot(
        dataset_test.x[::downsample],
        u_data[t_idx, 0],
        label="Data",
        color="black",
        linewidth=2.5,
        ls="--",
        alpha=0.6,
    )
    for model_type, u_pred in u_pred_dict.items():
        axes[0, i].plot(
            dataset_test.x[::downsample],
            u_pred[t_idx, 0],
            label=LABEL_DICT[model_type],
            color=COLOR_DICT[model_type],
            linewidth=1.5,
            alpha=0.8,
        )
        axes[1, i].plot(
            dataset_test.x[::downsample],
            u_pred[t_idx, 0] - u_data[t_idx, 0],
            label=LABEL_DICT[model_type],
            color=COLOR_DICT[model_type],
            linewidth=1.5,
            alpha=0.8,
        )
        axes[0, i].set_title("$t_{rollout}=$" + f"${t_idx * dt}$", fontsize="medium")
    axes[0, i].set_ylim((-1, 1))
    axes[1, i].set_ylim((-1, 1))
    # axes[0, i].text(0.4, -1.2, )